In [ ]:
import psycopg2

 

Connected successfully!


In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from Services.dbconnector import getconnection

print(getconnection())

<connection object at 0x000001CABE213EF0; dsn: 'user=postgres password=xxx dbname=smartretail host=localhost port=5432', closed: 0>


In [ ]:
import pandas as pd 

a = pd.read_csv ("../data/review.tsv" )
a.head()

In [ ]:
import importlib
import Services.ingestservice as ingestservice
importlib.reload(ingestservice)

from Services.ingestservice import ingestall
ingestall()
os.chdir(r"C:\Users\Siddhu\Desktop\personal\studying\Uplift\Project\SmartRetail")
print(os.getcwd()) 
 

el = ingestall()
print(el)

In [1]:
import os
os.chdir("..")  # move to SmartRetail/ root
print(os.getcwd())  # confirm

c:\Users\Siddhu\Desktop\personal\studying\Uplift\Project\SmartRetail


In [ ]:
import os
os.chdir("..")  # move to SmartRetail/ root
print(os.getcwd())  # confirm

"C:\Users\Siddhu\Desktop\personal\studying\Uplift\Project\SmartRetail\data\review.tsv"

c:\Users\Siddhu\Desktop


In [ ]:
import pandas as pd

cols = ["review_id", "product_id", "product_title",
        "star_rating", "review_headline", "review_body", "review_date"]

df = pd.read_csv(r"C:\Users\Siddhu\Desktop\personal\studying\Uplift\Project\SmartRetail\data\review.tsv", sep="\t", usecols=cols)
print(df.shape)
print(df.dtypes)
print(df["star_rating"].unique()[:20])

In [ ]:
import os
# os.chdir("..")
import pandas as pd 
from Services.ingestservice import getengine
from sqlalchemy import text

engine = getengine()
df= pd.read_sql("select * from bronze.sales" ,engine)
print(df.head(1))
df = df[~df["Invoice"].str.lower().str.startswith("c")]

df = df[ (df["Quantity"] >0)  & (df["Price"] > 0 )]
df = df.dropna(subset=["Customer ID","Description"])

df = df.drop_duplicates()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"]) 

df["Revenue"] = df["Quantity"] * df["Price"]
print(df.head())
df.shape

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.head())

In [ ]:
df['yearmonth'] = df["InvoiceDate"].dt.to_period('M').astype(str)
df1 = df.groupby("yearmonth").agg(
            Invoicecount =  ("Invoice" , 'nunique'),
            MonthlyRevenue = ("Revenue",'sum'),
            monAvgRev = ("Revenue",'mean')

        )

print(df1.shape)
print(df1.dtypes)
print(df1.head())

In [20]:
df1.to_sql("sales", getengine(), schema="gold", if_exists="replace", index=True)

25

In [ ]:
df= pd.read_sql("select * from bronze.reviews" ,engine)

df.head(1)


In [15]:
df= pd.read_sql("select * from bronze.reviews" ,engine) 
df = df.dropna(subset=["review_body"])
df = df.drop_duplicates()
df.to_sql("reviews",engine, schema="gold", if_exists="replace", index=False)
        

303

In [ ]:
import pandas as pd
from Services.ingestservice import getengine
from Services.transformservice import cleansales
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_sql("select * from gold.sales" ,getengine())
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(data["yearmonth"], data["MonthlyRevenue"], marker="o")
ax.set_xlabel("Month")
ax.set_ylabel("Total Revenue")
ax.set_title("Monthly Revenue Trend"    )
plt.xticks(rotation = 45)
plt.tight_layout()


In [ ]:
from Services.visualizeservice import plottopproducts, plotmonthlytrend, plotratingdist
import matplotlib.pyplot as plt

# fig1 = plottopproducts()
# plt.show()

fig2 = plotmonthlytrend()
plt.show()

fig3 = plotratingdist()
plt.show()

In [1]:
import os
os.chdir("..")  # move to SmartRetail/ root
print(os.getcwd())  # confirm

c:\Users\Siddhu\Desktop\personal\studying\Uplift\Project\SmartRetail


In [ ]:
from Services.embeddingservice import getreviewsforembedding
df = getreviewsforembedding()
print(df.shape)
print(df.head())

In [ ]:
from Services.embeddingservice import getreviewsforembedding ,generateembeddings
df = getreviewsforembedding()
sample = df.head(10)
embedded_df, vectors = generateembeddings(sample)
print(vectors.shape)  # should be (10, 384)

In [ ]:
from Services.embeddingservice import getreviewsforembedding ,generateembeddings ,build_faiss_index

df = getreviewsforembedding()
embedded_df, vectors = generateembeddings(df)
build_faiss_index(embedded_df, vectors)

In [ ]:
from Services.embeddingservice import searchreviews

results = searchreviews("gift card not working")
print(results[["review_id", "review_body"]])